# Week 01: PHASE 1 — Measurements, Units & Vectors — The Language of Motion

*Physics I (PHY101) . 3 Hours . Dr. Arif Solmaz*

## Learning Objectives

By the end of this week, you will be able to:

1. **Identify** and **convert** between SI base units and derived units
2. **Apply** dimensional analysis to verify equations and estimate quantities
3. **Use** significant figures correctly in calculations
4. **Decompose** vectors into components and reconstruct them
5. **Add vectors** graphically and algebraically
6. **Compute** dot products and vector projections
7. **Compute** cross products and interpret their geometric meaning
8. **Build** interactive Python visualizations of vector operations

## Core Mastery Connection

**PHASE 1 — "The Language of Motion":** Every prediction starts with the right units and vector decomposition. This week you build the foundational tools — SI units, dimensional analysis, and vector operations — that every future prediction depends on. Without correct units your equations are meaningless; without vector decomposition you cannot set up a single force or motion problem. Master these tools now and every week that follows becomes a direct application of them.

---
## 0. Setup

Run this cell first to import everything we need.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, Markdown

# Try ipywidgets -- works in Jupyter & Colab
try:
    import ipywidgets as widgets
    from ipywidgets import interact, interactive, FloatSlider, IntSlider, HBox, VBox
    WIDGETS_AVAILABLE = True
except ImportError:
    !pip install ipywidgets
    import ipywidgets as widgets
    from ipywidgets import interact, interactive, FloatSlider, IntSlider, HBox, VBox
    WIDGETS_AVAILABLE = True

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12
print('All imports successful!')

---
# Part 1: SI Units & Dimensional Analysis
---

## 1.1 The SI System -- Seven Base Units

Physics describes the universe using **measurable quantities**. The International System of Units (SI) defines **seven base units** from which all other units are derived.

| Quantity | Unit | Symbol | Everyday Analogy |
|---|---|---|---|
| Length | metre | m | A big step is about 1 m |
| Mass | kilogram | kg | A bag of sugar is about 1 kg |
| Time | second | s | One heartbeat is roughly 1 s |
| Electric current | ampere | A | Phone charger draws ~1--2 A |
| Temperature | kelvin | K | Room temp is about 295 K |
| Amount of substance | mole | mol | 6.022 x 10^23 particles |
| Luminous intensity | candela | cd | A candle is about 1 cd |

### SI Prefixes

| Prefix | Symbol | Factor | Example |
|---|---|---|---|
| giga | G | 10^9 | 1 GHz = 10^9 Hz |
| mega | M | 10^6 | 1 MW = 10^6 W |
| kilo | k | 10^3 | 1 km = 1000 m |
| centi | c | 10^-2 | 1 cm = 0.01 m |
| milli | m | 10^-3 | 1 mm = 0.001 m |
| micro | u | 10^-6 | 1 um = 10^-6 m |
| nano | n | 10^-9 | 1 nm = 10^-9 m |

## 1.2 Interactive Unit Converter

Use the sliders and dropdown to convert between common physics units.

In [ ]:
# ---- Interactive Unit Converter ----

conversion_db = {
    'Length': {
        'units': ['m', 'km', 'cm', 'mm', 'inch', 'ft', 'mile'],
        'to_base': [1, 1000, 0.01, 0.001, 0.0254, 0.3048, 1609.34]
    },
    'Mass': {
        'units': ['kg', 'g', 'mg', 'lb', 'oz'],
        'to_base': [1, 0.001, 1e-6, 0.453592, 0.0283495]
    },
    'Time': {
        'units': ['s', 'ms', 'min', 'hr', 'day'],
        'to_base': [1, 0.001, 60, 3600, 86400]
    },
    'Speed': {
        'units': ['m/s', 'km/h', 'mph', 'ft/s'],
        'to_base': [1, 1/3.6, 0.44704, 0.3048]
    }
}

category_dd = widgets.Dropdown(options=list(conversion_db.keys()), description='Category:')
from_dd = widgets.Dropdown(description='From:')
to_dd = widgets.Dropdown(description='To:')
value_input = widgets.FloatText(value=1.0, description='Value:')
result_label = widgets.HTML(value='<h3>Result: ---</h3>')

def update_units(*args):
    cat = category_dd.value
    from_dd.options = conversion_db[cat]['units']
    to_dd.options = conversion_db[cat]['units']

def do_conversion(*args):
    cat = category_dd.value
    db = conversion_db[cat]
    try:
        i_from = db['units'].index(from_dd.value)
        i_to = db['units'].index(to_dd.value)
        base_val = value_input.value * db['to_base'][i_from]
        result = base_val / db['to_base'][i_to]
        result_label.value = f'<h3>{value_input.value} {from_dd.value} = {result:.6g} {to_dd.value}</h3>'
    except (ValueError, ZeroDivisionError):
        result_label.value = '<h3>Result: ---</h3>'

category_dd.observe(update_units, names='value')
for w in [category_dd, from_dd, to_dd, value_input]:
    w.observe(do_conversion, names='value')

update_units()
do_conversion()

display(VBox([category_dd, HBox([from_dd, to_dd]), value_input, result_label]))

## 1.3 Dimensional Analysis

**Dimensional analysis** is a powerful technique: if an equation is physically correct, both sides must have the **same dimensions**.

We write dimensions using square brackets:
- Length: [L]
- Mass: [M]
- Time: [T]

### Rules
1. You can only **add or subtract** quantities with the **same dimensions**
2. Equations must be **dimensionally homogeneous**
3. Dimensional analysis can **check** your work or **guess** the form of an equation

### Example: Is v = v0 + at dimensionally correct?

- Left side: [v] = [L]/[T] = LT^-1
- Right side: [v0] + [a][t] = LT^-1 + (LT^-2)(T) = LT^-1 + LT^-1 = LT^-1
- Both sides match. The equation is dimensionally consistent.

## 1.4 Interactive Dimensional Analysis Checker

This tool lets you check whether simple physics formulas are dimensionally consistent.

In [ ]:
# ---- Dimensional Analysis Checker ----

# We represent dimensions as (L_power, M_power, T_power)
known_dims = {
    'x': (1, 0, 0),    # position [L]
    'd': (1, 0, 0),    # distance [L]
    'v': (1, 0, -1),   # velocity [LT^-1]
    'v0': (1, 0, -1),  # initial velocity
    'a': (1, 0, -2),   # acceleration [LT^-2]
    'g': (1, 0, -2),   # gravitational acceleration
    't': (0, 0, 1),    # time [T]
    'F': (1, 1, -2),   # force [MLT^-2]
    'm': (0, 1, 0),    # mass [M]
    'E': (2, 1, -2),   # energy [ML^2T^-2]
    'P': (2, 1, -3),   # power [ML^2T^-3]
    'p': (1, 1, -1),   # momentum [MLT^-1]
}

def dim_str(d):
    parts = []
    labels = ['L', 'M', 'T']
    for i, lab in enumerate(labels):
        if d[i] == 1:
            parts.append(lab)
        elif d[i] != 0:
            parts.append(f'{lab}^{d[i]}')
    return ' '.join(parts) if parts else '1 (dimensionless)'

print("Known variables and their dimensions:")
print("=" * 45)
for var, d in known_dims.items():
    print(f"  {var:>3s}  -->  [{dim_str(d)}]")

print("\n--- Quick Check Examples ---")
# Check: E = 0.5 * m * v^2
lhs = known_dims['E']  # (2, 1, -2)
# m * v^2 = (0,1,0) + 2*(1,0,-1) = (2,1,-2)
rhs = tuple(known_dims['m'][i] + 2*known_dims['v'][i] for i in range(3))
print(f"E = 1/2 m v^2 :  LHS = [{dim_str(lhs)}],  RHS = [{dim_str(rhs)}]  -->  {'CONSISTENT' if lhs == rhs else 'INCONSISTENT'}")

# Check: v = a * t^2  (WRONG!)
lhs2 = known_dims['v']  # (1, 0, -1)
rhs2 = tuple(known_dims['a'][i] + 2*known_dims['t'][i] for i in range(3))  # (1,0,-2) + (0,0,2) = (1,0,0)
print(f"v = a * t^2   :  LHS = [{dim_str(lhs2)}],  RHS = [{dim_str(rhs2)}]  -->  {'CONSISTENT' if lhs2 == rhs2 else 'INCONSISTENT'}")

## 1.5 Significant Figures

**Significant figures** tell us how precisely a number is known.

### Rules for Counting Significant Figures

| Rule | Example | Sig Figs |
|---|---|---|
| All non-zero digits count | 324.7 | 4 |
| Zeros between non-zero digits count | 3007 | 4 |
| Leading zeros do NOT count | 0.0045 | 2 |
| Trailing zeros after decimal count | 2.500 | 4 |
| Trailing zeros without decimal are ambiguous | 1500 | 2, 3, or 4 |

### Rules for Calculations
- **Multiplication/Division**: result has same number of sig figs as the **least precise** input
- **Addition/Subtraction**: result has same number of **decimal places** as the least precise input

---
# Part 2: Vectors
---

## 2.1 Scalars vs. Vectors

| Property | Scalar | Vector |
|---|---|---|
| Has magnitude? | Yes | Yes |
| Has direction? | No | Yes |
| Examples | mass, temperature, time | displacement, velocity, force |
| Notation | m, T, t | **v**, **F** (bold) or with arrow |

### Vector Components

Any 2D vector **A** can be written in terms of components:

**A** = Ax **i** + Ay **j**

where:
- Ax = |A| cos(theta) -- the x-component
- Ay = |A| sin(theta) -- the y-component
- |A| = sqrt(Ax^2 + Ay^2) -- the magnitude
- theta = arctan(Ay / Ax) -- the direction angle (from +x axis)

## 2.2 Interactive Vector Components Visualizer

Use the sliders to change the magnitude and angle of a vector. Watch how the components change.

In [ ]:
# ---- Vector Component Visualizer ----

def plot_vector_components(magnitude=5.0, angle_deg=45.0):
    angle_rad = np.radians(angle_deg)
    Ax = magnitude * np.cos(angle_rad)
    Ay = magnitude * np.sin(angle_rad)
    
    fig, ax = plt.subplots(1, 1, figsize=(7, 7))
    lim = 12
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title('Vector Components', fontsize=14)
    
    # Draw the main vector
    ax.annotate('', xy=(Ax, Ay), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='blue', lw=2.5))
    ax.text(Ax/2 - 0.8, Ay/2 + 0.5, f'A = {magnitude:.1f}',
            color='blue', fontsize=13, fontweight='bold')
    
    # Draw x-component (dashed)
    ax.annotate('', xy=(Ax, 0), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='red', lw=2, linestyle='--'))
    ax.text(Ax/2, -1.0, f'Ax = {Ax:.2f}', color='red', fontsize=12, ha='center')
    
    # Draw y-component (dashed)
    ax.annotate('', xy=(Ax, Ay), xytext=(Ax, 0),
                arrowprops=dict(arrowstyle='->', color='green', lw=2, linestyle='--'))
    ax.text(Ax + 0.8, Ay/2, f'Ay = {Ay:.2f}', color='green', fontsize=12)
    
    # Draw angle arc
    theta_range = np.linspace(0, angle_rad, 50)
    r_arc = min(2.0, magnitude * 0.3)
    ax.plot(r_arc * np.cos(theta_range), r_arc * np.sin(theta_range), 'purple', lw=1.5)
    ax.text(r_arc * 1.3 * np.cos(angle_rad/2), r_arc * 1.3 * np.sin(angle_rad/2),
            f'{angle_deg:.0f} deg', color='purple', fontsize=11)
    
    plt.tight_layout()
    plt.show()

interact(plot_vector_components,
         magnitude=FloatSlider(min=0.5, max=10.0, step=0.5, value=5.0, description='|A|:'),
         angle_deg=FloatSlider(min=0, max=360, step=5, value=45, description='Angle (deg):'));

## 2.3 Vector Addition

To add two vectors **A** and **B**:

**Graphical method (tip-to-tail):**
1. Draw **A**
2. Place the tail of **B** at the tip of **A**
3. The resultant **R** goes from the tail of **A** to the tip of **B**

**Component method (algebraic):**
- Rx = Ax + Bx
- Ry = Ay + By
- |R| = sqrt(Rx^2 + Ry^2)
- theta_R = arctan(Ry / Rx)

## 2.4 Interactive Vector Addition Visualizer

Adjust both vectors using sliders and see how the resultant changes in real time.

In [ ]:
# ---- Interactive Vector Addition ----

def plot_vector_addition(magA=4.0, angleA=30.0, magB=3.0, angleB=120.0):
    angA_rad = np.radians(angleA)
    angB_rad = np.radians(angleB)
    
    Ax, Ay = magA * np.cos(angA_rad), magA * np.sin(angA_rad)
    Bx, By = magB * np.cos(angB_rad), magB * np.sin(angB_rad)
    Rx, Ry = Ax + Bx, Ay + By
    magR = np.sqrt(Rx**2 + Ry**2)
    angR = np.degrees(np.arctan2(Ry, Rx))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    for ax in [ax1, ax2]:
        lim = 12
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        ax.axhline(0, color='k', lw=0.5)
        ax.axvline(0, color='k', lw=0.5)
    
    # Left plot: tip-to-tail method
    ax1.set_title('Tip-to-Tail Method', fontsize=13)
    ax1.annotate('', xy=(Ax, Ay), xytext=(0, 0),
                 arrowprops=dict(arrowstyle='->', color='blue', lw=2.5))
    ax1.text(Ax/2 - 0.3, Ay/2 + 0.5, 'A', color='blue', fontsize=14, fontweight='bold')
    
    ax1.annotate('', xy=(Ax + Bx, Ay + By), xytext=(Ax, Ay),
                 arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
    ax1.text(Ax + Bx/2 + 0.3, Ay + By/2 + 0.3, 'B', color='red', fontsize=14, fontweight='bold')
    
    ax1.annotate('', xy=(Rx, Ry), xytext=(0, 0),
                 arrowprops=dict(arrowstyle='->', color='green', lw=3))
    ax1.text(Rx/2 - 1.2, Ry/2 - 0.8, f'R = {magR:.2f}', color='green', fontsize=13, fontweight='bold')
    
    # Right plot: component breakdown
    ax2.set_title('Component Method', fontsize=13)
    ax2.annotate('', xy=(Ax, Ay), xytext=(0, 0),
                 arrowprops=dict(arrowstyle='->', color='blue', lw=2))
    ax2.annotate('', xy=(Bx, By), xytext=(0, 0),
                 arrowprops=dict(arrowstyle='->', color='red', lw=2))
    ax2.annotate('', xy=(Rx, Ry), xytext=(0, 0),
                 arrowprops=dict(arrowstyle='->', color='green', lw=3))
    
    # Dashed component lines for R
    ax2.plot([0, Rx], [0, 0], 'g--', alpha=0.5)
    ax2.plot([Rx, Rx], [0, Ry], 'g--', alpha=0.5)
    
    info = (f'A = ({Ax:.2f}, {Ay:.2f})\n'
            f'B = ({Bx:.2f}, {By:.2f})\n'
            f'R = ({Rx:.2f}, {Ry:.2f})\n'
            f'|R| = {magR:.2f}\n'
            f'angle_R = {angR:.1f} deg')
    ax2.text(-lim + 0.5, lim - 1.5, info, fontsize=10, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

interact(plot_vector_addition,
         magA=FloatSlider(min=0.5, max=10, step=0.5, value=4, description='|A|:'),
         angleA=FloatSlider(min=0, max=360, step=5, value=30, description='A angle:'),
         magB=FloatSlider(min=0.5, max=10, step=0.5, value=3, description='|B|:'),
         angleB=FloatSlider(min=0, max=360, step=5, value=120, description='B angle:'));

## 2.5 Dot Product and Vector Projection

The **dot product** (scalar product) of two vectors measures how much they point in the same direction:

**A** . **B** = |A| |B| cos(theta) = Ax*Bx + Ay*By

Key properties:
- If theta = 0 deg (parallel): A . B = |A||B| (maximum)
- If theta = 90 deg (perpendicular): A . B = 0
- If theta = 180 deg (anti-parallel): A . B = -|A||B|

### Vector Projection

The **projection** of **A** onto **B** gives the component of **A** in the direction of **B**:

- Scalar projection: comp_B(A) = (A . B) / |B|
- Vector projection: proj_B(A) = [(A . B) / |B|^2] * **B**

## 2.6 Interactive Projection Demo

See how the projection of **A** onto **B** changes as you rotate the vectors.

In [ ]:
# ---- Interactive Vector Projection ----

def plot_projection(magA=5.0, angleA=60.0, magB=6.0, angleB=10.0):
    aA = np.radians(angleA)
    aB = np.radians(angleB)
    
    A = np.array([magA * np.cos(aA), magA * np.sin(aA)])
    B = np.array([magB * np.cos(aB), magB * np.sin(aB)])
    
    # Dot product
    dot = np.dot(A, B)
    magB_val = np.linalg.norm(B)
    
    # Projection of A onto B
    scalar_proj = dot / magB_val
    B_hat = B / magB_val
    vector_proj = scalar_proj * B_hat
    
    fig, ax = plt.subplots(figsize=(8, 8))
    lim = 12
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', lw=0.5)
    ax.axvline(0, color='k', lw=0.5)
    ax.set_title('Vector Projection of A onto B', fontsize=14)
    
    # Draw B
    ax.annotate('', xy=B, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
    ax.text(B[0] + 0.3, B[1] + 0.3, 'B', color='red', fontsize=14, fontweight='bold')
    
    # Draw A
    ax.annotate('', xy=A, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='blue', lw=2.5))
    ax.text(A[0] + 0.3, A[1] + 0.3, 'A', color='blue', fontsize=14, fontweight='bold')
    
    # Draw projection vector
    ax.annotate('', xy=vector_proj, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='green', lw=3))
    ax.text(vector_proj[0]/2 - 1.0, vector_proj[1]/2 - 1.0,
            f'proj = {scalar_proj:.2f}', color='green', fontsize=12, fontweight='bold')
    
    # Dashed perpendicular line from tip of A to projection
    ax.plot([A[0], vector_proj[0]], [A[1], vector_proj[1]], 'k--', alpha=0.5, lw=1)
    
    # Right angle marker
    angle_between = np.degrees(np.arccos(np.clip(dot / (np.linalg.norm(A) * magB_val), -1, 1)))
    
    info = (f'A . B = {dot:.2f}\n'
            f'Angle between = {angle_between:.1f} deg\n'
            f'Scalar projection = {scalar_proj:.2f}\n'
            f'|proj vector| = {abs(scalar_proj):.2f}')
    ax.text(-lim + 0.5, lim - 0.5, info, fontsize=10, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

interact(plot_projection,
         magA=FloatSlider(min=1, max=10, step=0.5, value=5, description='|A|:'),
         angleA=FloatSlider(min=0, max=360, step=5, value=60, description='A angle:'),
         magB=FloatSlider(min=1, max=10, step=0.5, value=6, description='|B|:'),
         angleB=FloatSlider(min=0, max=360, step=5, value=10, description='B angle:'));

## 2.7 Cross Product

The **cross product** (vector product) of two vectors produces a **new vector** that is perpendicular to both inputs:

**A** x **B** = |A| |B| sin(theta) **n**

where **n** is a unit vector perpendicular to the plane of **A** and **B**, determined by the **right-hand rule**: curl the fingers of your right hand from **A** toward **B**, and your thumb points in the direction of **A** x **B**.

### Key Properties

- **Anti-commutative:** **A** x **B** = -(**B** x **A**) (order matters!)
- **Perpendicular:** The result is always perpendicular to both **A** and **B**
- **Parallel vectors:** If **A** || **B** (theta = 0 or 180 deg), then **A** x **B** = **0**
- **Perpendicular vectors:** If **A** perp **B** (theta = 90 deg), then |**A** x **B**| = |A||B| (maximum)

### 2D Simplification

For two vectors in the xy-plane, **A** = (Ax, Ay) and **B** = (Bx, By), the cross product has only a z-component:

(**A** x **B**)_z = Ax * By - Ay * Bx

This scalar gives the **signed magnitude**: positive means **n** points out of the page (+z), negative means into the page (-z).

### 3D Component Formula

For 3D vectors **A** = (Ax, Ay, Az) and **B** = (Bx, By, Bz):

**A** x **B** = (Ay*Bz - Az*By) **i** - (Ax*Bz - Az*Bx) **j** + (Ax*By - Ay*Bx) **k**

This can be remembered as the determinant:

|  **i**   **j**   **k**  |
|  Ax      Ay      Az     |
|  Bx      By      Bz     |

### Physical Meaning

The magnitude |**A** x **B**| = |A||B|sin(theta) equals the **area of the parallelogram** formed by **A** and **B**. This geometric interpretation appears throughout physics: torque, angular momentum, and magnetic force all involve the cross product because they depend on the *perpendicular* component of one vector relative to another.

In [ ]:
# ---- Cross Product: Worked Example ----

# --- 2D Cross Product ---
print("=== 2D Cross Product ===")
A_2d = np.array([3, 2])
B_2d = np.array([1, 4])

cross_2d = A_2d[0] * B_2d[1] - A_2d[1] * B_2d[0]
print(f"A = {A_2d}")
print(f"B = {B_2d}")
print(f"A x B = Ax*By - Ay*Bx = {A_2d[0]}*{B_2d[1]} - {A_2d[1]}*{B_2d[0]} = {cross_2d}")
print(f"Magnitude = {abs(cross_2d)}, direction = {'out of page (+z)' if cross_2d > 0 else 'into page (-z)'}")
print(f"Parallelogram area = {abs(cross_2d)}")

# --- 3D Cross Product ---
print("\n=== 3D Cross Product ===")
A_3d = np.array([2, 3, 1])
B_3d = np.array([1, -1, 2])

C_3d = np.cross(A_3d, B_3d)
print(f"A = {A_3d}")
print(f"B = {B_3d}")
print(f"A x B = {C_3d}")
print(f"|A x B| = {np.linalg.norm(C_3d):.4f}")

# Verify perpendicularity
print(f"\nVerification (should be zero):")
print(f"  (A x B) . A = {np.dot(C_3d, A_3d)}")
print(f"  (A x B) . B = {np.dot(C_3d, B_3d)}")

# Verify anti-commutativity
print(f"\nAnti-commutativity:")
print(f"  A x B = {np.cross(A_3d, B_3d)}")
print(f"  B x A = {np.cross(B_3d, A_3d)}  (opposite sign!)")

# --- 3D Visualization ---
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

origin = np.array([0, 0, 0])

# Draw vectors as quivers
ax.quiver(*origin, *A_3d, color='blue', linewidth=2.5, arrow_length_ratio=0.1, label='A = (2, 3, 1)')
ax.quiver(*origin, *B_3d, color='red', linewidth=2.5, arrow_length_ratio=0.1, label='B = (1, -1, 2)')
ax.quiver(*origin, *C_3d, color='green', linewidth=2.5, arrow_length_ratio=0.1, label=f'A x B = {tuple(C_3d)}')

# Draw parallelogram formed by A and B
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
verts = [list(zip([0, A_3d[0], A_3d[0]+B_3d[0], B_3d[0]],
                  [0, A_3d[1], A_3d[1]+B_3d[1], B_3d[1]],
                  [0, A_3d[2], A_3d[2]+B_3d[2], B_3d[2]]))]
poly = Poly3DCollection(verts, alpha=0.15, facecolor='purple', edgecolor='purple', linewidth=1)
ax.add_collection3d(poly)

# Labels at vector tips
ax.text(*A_3d, '  A', color='blue', fontsize=12, fontweight='bold')
ax.text(*B_3d, '  B', color='red', fontsize=12, fontweight='bold')
ax.text(*C_3d, '  A x B', color='green', fontsize=12, fontweight='bold')

# Set axis limits and labels
max_val = max(np.max(np.abs(A_3d)), np.max(np.abs(B_3d)), np.max(np.abs(C_3d))) + 1
ax.set_xlim([-max_val, max_val])
ax.set_ylim([-max_val, max_val])
ax.set_zlim([-max_val, max_val])
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_title('Cross Product: A x B is Perpendicular to Both A and B', fontsize=13)
ax.legend(loc='upper left', fontsize=10)
ax.view_init(elev=20, azim=35)

plt.tight_layout()
plt.show()

> **Looking ahead:** We will use the cross product when we study torque (Week 8) and angular momentum (Week 10). In Physics II, it appears in magnetic force calculations (**F** = q**v** x **B**).

## 2.9 Animated Vector Addition (Tip-to-Tail)

Watch how vectors are added one by one using the tip-to-tail method.

In [ ]:
# ---- Animated Tip-to-Tail Vector Addition ----

vectors = [
    (3.0, 30),    # vector 1: magnitude 3, angle 30 deg
    (4.0, 110),   # vector 2: magnitude 4, angle 110 deg
    (2.5, 200),   # vector 3: magnitude 2.5, angle 200 deg
    (3.5, 320),   # vector 4: magnitude 3.5, angle 320 deg
]
colors = ['blue', 'red', 'orange', 'purple']

# Precompute cumulative positions
positions = [(0, 0)]
for mag, ang in vectors:
    px, py = positions[-1]
    dx = mag * np.cos(np.radians(ang))
    dy = mag * np.sin(np.radians(ang))
    positions.append((px + dx, py + dy))

fig, ax = plt.subplots(figsize=(8, 8))
lim = 10
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='k', lw=0.5)
ax.axvline(0, color='k', lw=0.5)
ax.set_title('Tip-to-Tail Vector Addition (Animated)', fontsize=13)

arrows = []

def init():
    return []

def animate(frame):
    # Remove old artists
    for a in arrows:
        a.remove()
    arrows.clear()
    
    n_show = min(frame + 1, len(vectors))
    for i in range(n_show):
        sx, sy = positions[i]
        mag, ang = vectors[i]
        dx = mag * np.cos(np.radians(ang))
        dy = mag * np.sin(np.radians(ang))
        arrow = ax.annotate('', xy=(sx + dx, sy + dy), xytext=(sx, sy),
                    arrowprops=dict(arrowstyle='->', color=colors[i], lw=2.5))
        arrows.append(arrow)
    
    # Show resultant after all vectors
    if frame >= len(vectors):
        rx, ry = positions[-1]
        resultant_arrow = ax.annotate('', xy=(rx, ry), xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color='green', lw=3.5))
        arrows.append(resultant_arrow)
    
    return arrows

anim = FuncAnimation(fig, animate, init_func=init,
                     frames=len(vectors) + 2, interval=1000,
                     blit=False, repeat=True)
plt.close(fig)
HTML(anim.to_jshtml())

---
# Part 3: Worked Examples
---

## Worked Example 1: Dimensional Analysis

**Problem:** The period T of a pendulum depends on its length L and gravitational acceleration g.
Use dimensional analysis to find how T depends on L and g.

**Solution:**
Assume T = k * L^a * g^b (where k is a dimensionless constant)

Dimensions:
- [T] = T^1
- [L^a] = L^a
- [g^b] = (L T^-2)^b = L^b T^(-2b)

Setting up equations:
- For L: 0 = a + b  -->  a = -b
- For T: 1 = -2b    -->  b = -1/2, so a = 1/2

Therefore: T = k * sqrt(L/g)

(The exact result is T = 2*pi*sqrt(L/g), so k = 2*pi)

In [ ]:
# Verify with numbers
L = 1.0   # 1 metre pendulum
g = 9.81  # m/s^2
T = 2 * np.pi * np.sqrt(L / g)
print(f"Pendulum length: {L} m")
print(f"Period: T = 2*pi*sqrt(L/g) = {T:.3f} s")
print(f"That's about {T:.1f} seconds per swing -- try it with a 1m string!")

## Worked Example 2: Vector Addition

**Problem:** A hiker walks 5 km at 37 deg North of East, then 3 km due North. Find the resultant displacement.

**Solution:**

In [ ]:
# Worked Example: Hiker displacement

# Step 1: Decompose each leg into components
# Leg 1: 5 km at 37 deg above +x axis
mag1, ang1 = 5.0, 37.0
A1x = mag1 * np.cos(np.radians(ang1))
A1y = mag1 * np.sin(np.radians(ang1))
print(f"Leg 1: ({A1x:.2f}, {A1y:.2f}) km")

# Leg 2: 3 km due North (90 deg)
mag2, ang2 = 3.0, 90.0
A2x = mag2 * np.cos(np.radians(ang2))
A2y = mag2 * np.sin(np.radians(ang2))
print(f"Leg 2: ({A2x:.2f}, {A2y:.2f}) km")

# Step 2: Add components
Rx = A1x + A2x
Ry = A1y + A2y
print(f"\nResultant components: ({Rx:.2f}, {Ry:.2f}) km")

# Step 3: Find magnitude and direction
R_mag = np.sqrt(Rx**2 + Ry**2)
R_ang = np.degrees(np.arctan2(Ry, Rx))
print(f"Resultant magnitude: |R| = {R_mag:.2f} km")
print(f"Resultant direction: {R_ang:.1f} deg from East")

# Plot it
fig, ax = plt.subplots(figsize=(7, 7))
ax.set_xlim(-1, 7)
ax.set_ylim(-1, 8)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_xlabel('East (km)')
ax.set_ylabel('North (km)')
ax.set_title('Hiker Displacement', fontsize=14)

ax.annotate('', xy=(A1x, A1y), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='blue', lw=2.5))
ax.text(A1x/2 + 0.2, A1y/2 - 0.5, '5 km', color='blue', fontsize=12)

ax.annotate('', xy=(A1x + A2x, A1y + A2y), xytext=(A1x, A1y),
            arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
ax.text(A1x + 0.3, A1y + A2y/2, '3 km', color='red', fontsize=12)

ax.annotate('', xy=(Rx, Ry), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='green', lw=3))
ax.text(Rx/2 - 1.5, Ry/2, f'R = {R_mag:.2f} km', color='green', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## Worked Example 3: Dot Product and Projection

**Problem:** Two forces act on an object: **F1** = (3, 4) N and **F2** = (5, -2) N.
Find:
- (a) The dot product F1 . F2
- (b) The angle between the forces
- (c) The projection of F1 onto F2

In [ ]:
F1 = np.array([3, 4])
F2 = np.array([5, -2])

# (a) Dot product
dot = np.dot(F1, F2)
print(f"(a) F1 . F2 = {F1[0]}*{F2[0]} + {F1[1]}*{F2[1]} = {dot}")

# (b) Angle between them
mag_F1 = np.linalg.norm(F1)
mag_F2 = np.linalg.norm(F2)
cos_theta = dot / (mag_F1 * mag_F2)
theta = np.degrees(np.arccos(cos_theta))
print(f"(b) |F1| = {mag_F1:.2f}, |F2| = {mag_F2:.2f}")
print(f"    cos(theta) = {cos_theta:.4f}")
print(f"    theta = {theta:.1f} degrees")

# (c) Projection of F1 onto F2
scalar_proj = dot / mag_F2
F2_hat = F2 / mag_F2
vector_proj = scalar_proj * F2_hat
print(f"(c) Scalar projection = {scalar_proj:.2f}")
print(f"    Vector projection = ({vector_proj[0]:.2f}, {vector_proj[1]:.2f}) N")

---
## Problem Set

**Core Mastery Workflow — For each problem: Draw the diagram -> Identify the principle -> Write the equation -> Predict -> Verify.**

**Instructions:** Solve each problem in the code cell provided. Show your work using Python calculations. Use `numpy` functions where appropriate. Problems are graded by difficulty:
- **L1 (Basic):** Single-concept, straightforward calculation
- **L2 (Intermediate):** Multi-step, combines two or more concepts
- **L3 (Challenge):** Multi-concept integration, deeper analysis required

### L1 -- P1: Unit Conversion

A mechatronics sensor datasheet specifies a measurement range of 2.50 mm with a resolution of 0.80 μm. Convert both values to metres. Express the number of distinguishable levels (range / resolution) as a dimensionless integer.

<details><summary>Answer</summary>Range = 2.50 × 10⁻³ m, Resolution = 8.0 × 10⁻⁷ m, Levels = 3125</details>

In [ ]:
# ✏️ [P1] Your solution here


### L1 -- P2: Dimensional Analysis Check

A student proposes the formula for the drag force on a sphere: F = C · ρ · v² · A, where C is a dimensionless drag coefficient, ρ is fluid density (kg/m³), v is speed (m/s), and A is cross-sectional area (m²). Verify that this expression has dimensions of force [MLT⁻²].

<details><summary>Answer</summary>[ρ v² A] = (kg/m³)(m/s)²(m²) = kg·m⁻³·m²·s⁻²·m² = kg·m·s⁻² = [MLT⁻²]. Consistent with force.</details>

In [ ]:
# ✏️ [P2] Your solution here


### L1 -- P3: Vector Components

A force of magnitude 85 N acts at 53° above the positive x-axis. Decompose this force into its x- and y-components.

<details><summary>Answer</summary>Fx = 85 cos(53°) = 51.2 N, Fy = 85 sin(53°) = 67.9 N</details>

In [ ]:
# ✏️ [P3] Your solution here


### L1 -- P4: Vector Magnitude and Direction

A displacement vector has components dx = −12.0 m and dy = 5.0 m. Find the magnitude of the displacement and the angle it makes with the positive x-axis (measured counterclockwise).

<details><summary>Answer</summary>|d| = √(144 + 25) = 13.0 m, θ = arctan(5/−12) = 180° − 22.6° = 157.4°</details>

In [ ]:
# ✏️ [P4] Your solution here


### L2 -- P5: Multi-Vector Addition

A robot arm moves through three successive displacements: **d1** = 0.40 m at 30°, **d2** = 0.60 m at 150°, and **d3** = 0.35 m at 270°. Find the resultant displacement vector (magnitude and direction) using the component method.

<details><summary>Answer</summary>Rx = 0.40cos30° + 0.60cos150° + 0.35cos270° = 0.346 − 0.520 + 0 = −0.174 m; Ry = 0.40sin30° + 0.60sin150° + 0.35sin270° = 0.200 + 0.300 − 0.350 = 0.150 m; |R| = 0.230 m at 139.3°</details>

In [ ]:
# ✏️ [P5] Your solution here


### L2 -- P6: Dot Product Application

Two cables pull on a ring. Cable A exerts **F_A** = (120, 200) N and cable B exerts **F_B** = (250, −80) N. (a) Find the angle between the two force vectors. (b) Calculate the component of **F_A** in the direction of **F_B**.

<details><summary>Answer</summary>(a) F_A · F_B = 30000 − 16000 = 14000 N²; |F_A| = 233.2 N, |F_B| = 262.5 N; θ = arccos(14000/(233.2×262.5)) = 76.8°. (b) Scalar projection = 14000/262.5 = 53.3 N</details>

In [ ]:
# ✏️ [P6] Your solution here


### L2 -- P7: Cross Product and Area

Two edges of a triangular sail are defined by vectors **A** = (3.0, 1.0, 0) m and **B** = (1.0, 4.0, 0) m originating from the same corner. (a) Compute **A** × **B**. (b) Find the area of the triangle.

<details><summary>Answer</summary>(a) A × B = (0, 0, 3×4 − 1×1) = (0, 0, 11) m², direction: +z (out of page). (b) Parallelogram area = 11 m², triangle area = 5.5 m²</details>

In [ ]:
# ✏️ [P7] Your solution here


### L2 -- P8: Dimensional Analysis Derivation

The speed of waves on a stretched string depends on the tension T (in newtons) and the linear mass density μ (in kg/m). Use dimensional analysis to derive how the wave speed v depends on T and μ. Write v = k · T^a · μ^b and solve for a and b.

<details><summary>Answer</summary>[v] = LT⁻¹; [T] = MLT⁻²; [μ] = ML⁻¹. Setting up: L: 1 = a − b, M: 0 = a + b, T: −1 = −2a. So a = 1/2, b = −1/2. v = k√(T/μ).</details>

In [ ]:
# ✏️ [P8] Your solution here


### L3 -- P9: 3D Cross Product and Torque Preview

A wrench handle lies along the vector **r** = (0.25, 0, 0) m (from the bolt to where you grip). You push with force **F** = (0, 0, −80) N (straight down in the z-direction). (a) Compute the torque τ = **r** × **F**. (b) Find the magnitude of the torque. (c) If you could only push with the same 80 N magnitude but wanted to maximize the torque, what direction should you push?

<details><summary>Answer</summary>(a) τ = r × F = (0·(−80) − 0·0, 0·0 − 0.25·(−80), 0.25·0 − 0·0) = (0, 20, 0) N·m. (b) |τ| = 20 N·m. (c) Push perpendicular to r in the yz-plane, e.g., F = (0, 0, −80) or (0, −80, 0) both give 20 N·m; any direction perpendicular to r maximizes torque.</details>

In [ ]:
# ✏️ [P9] Your solution here


### L3 -- P10: Navigation with Vectors and Unit Conversion

A drone starts at the origin and executes four waypoints in sequence: (1) fly 1.20 km at bearing 045° (NE), (2) fly 800 m at bearing 180° (S), (3) fly 1.50 km at bearing 300°, (4) fly 0.60 km at bearing 090° (E). Note: bearing is measured clockwise from North. (a) Convert all distances to metres and bearings to standard math angles (CCW from +x). (b) Find the resultant displacement magnitude in metres. (c) What single bearing would take the drone directly from start to end?

<details><summary>Answer</summary>(a) Bearing → math angle: θ_math = 90° − bearing. Leg 1: 1200 m at 45°, Leg 2: 800 m at −90° (or 270°), Leg 3: 1500 m at 150°, Leg 4: 600 m at 0°. (b) Rx = 1200cos45 + 800cos270 + 1500cos150 + 600cos0 = 848.5 + 0 − 1299.0 + 600 = 149.5 m; Ry = 848.5 + (−800) + 750 + 0 = 798.5 m; |R| = 812.4 m. (c) Math angle = arctan(798.5/149.5) = 79.4°, bearing = 90 − 79.4 = 10.6° (roughly NNE).</details>

In [ ]:
# ✏️ [P10] Your solution here


---
## Bridge to Next Week

This week we learned the **language** of physics: units, dimensions, and vectors. These are the tools we will use every single week.

**Next week** we start using these tools to describe **motion**: How do objects move in one dimension? We will define:
- **Position** x(t)
- **Velocity** v(t) = dx/dt
- **Acceleration** a(t) = dv/dt

The vectors we learned today will become essential when we extend to 2D motion in later weeks.

**Preparation:** Review how to read x-t and v-t graphs. Think about: if you know the velocity at every instant, can you figure out the position?